In [ ]:

import pandas as pd 
import numpy as np 
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import joblib




In [ ]:
df =pd.read_csv('D:\\Dossiers\\WaterTracker_Project\\watertracker-backend\\data\\historique\\ndwi_sources_historique.csv')
df.head()

,source_id,longitude,latitude,periode,saison,debut,fin,ndwi_moyen,ndvi_moyen
0,1,-1.853216,12.095082,2020-S1,seche,2020-01-01,2020-06-30,-0.187815,0.084677
1,1,-1.869867,12.184925,2020-S1,seche,2020-01-01,2020-06-30,0.057975,-0.081340
2,1,-1.924281,12.414088,2020-S1,seche,2020-01-01,2020-06-30,-0.032191,-0.011703
3,1,-1.925239,12.413330,2020-S1,seche,2020-01-01,2020-06-30,-0.022204,-0.029976
4,1,-1.926437,12.413010,2020-S1,seche,2020-01-01,2020-06-30,-0.026483,-0.033256


In [ ]:
df.source_id.values_counts()

In [22]:
df.shape

(2780, 9)

In [23]:
df = df.dropna(subset=['ndwi_moyen'])

In [24]:
df.head()

,source_id,longitude,latitude,periode,saison,debut,fin,ndwi_moyen,ndvi_moyen
0,1,-1.853216,12.095082,2020-S1,seche,2020-01-01,2020-06-30,-0.187815,0.084677
1,1,-1.869867,12.184925,2020-S1,seche,2020-01-01,2020-06-30,0.057975,-0.081340
2,1,-1.924281,12.414088,2020-S1,seche,2020-01-01,2020-06-30,-0.032191,-0.011703
3,1,-1.925239,12.413330,2020-S1,seche,2020-01-01,2020-06-30,-0.022204,-0.029976
4,1,-1.926437,12.413010,2020-S1,seche,2020-01-01,2020-06-30,-0.026483,-0.033256


In [25]:
df.shape

(2780, 9)

# 2. Encodage des variables categorielles

In [26]:
le_saison   = LabelEncoder()
df['saison_encoded'] = le_saison.fit_transform(df['saison'])
df['annee']          = df['periode'].str[:4].astype(int)
df['semestre']       = df['periode'].str[-1].astype(int)
df.head()

,source_id,longitude,latitude,periode,saison,debut,fin,ndwi_moyen,ndvi_moyen,saison_encoded,annee,semestre
0,1,-1.853216,12.095082,2020-S1,seche,2020-01-01,2020-06-30,-0.187815,0.084677,1,2020,1
1,1,-1.869867,12.184925,2020-S1,seche,2020-01-01,2020-06-30,0.057975,-0.081340,1,2020,1
2,1,-1.924281,12.414088,2020-S1,seche,2020-01-01,2020-06-30,-0.032191,-0.011703,1,2020,1
3,1,-1.925239,12.413330,2020-S1,seche,2020-01-01,2020-06-30,-0.022204,-0.029976,1,2020,1
4,1,-1.926437,12.413010,2020-S1,seche,2020-01-01,2020-06-30,-0.026483,-0.033256,1,2020,1


# 3. Feature engineering PAR SOURCE


In [27]:
df = df.sort_values(['source_id', 'periode'])

df['ndwi_t1']        = df.groupby('source_id')['ndwi_moyen'].shift(1)
df['ndwi_t2']        = df.groupby('source_id')['ndwi_moyen'].shift(2)
df['ndwi_t3']        = df.groupby('source_id')['ndwi_moyen'].shift(3)
df['tendance']       = df['ndwi_moyen'] - df['ndwi_t1']
df['tendance_long']  = df['ndwi_moyen'] - df['ndwi_t3']
df['ndwi_moy_src']   = df.groupby('source_id')['ndwi_moyen'].transform('mean')
df['ndwi_futur']     = df.groupby('source_id')['ndwi_moyen'].shift(-1)
df.head()

,source_id,longitude,latitude,periode,saison,debut,fin,ndwi_moyen,ndvi_moyen,saison_encoded,annee,semestre,ndwi_t1,ndwi_t2,ndwi_t3,tendance,tendance_long,ndwi_moy_src,ndwi_futur
0,1,-1.853216,12.095082,2020-S1,seche,2020-01-01,2020-06-30,-0.187815,0.084677,1,2020,1,NaN,NaN,NaN,NaN,NaN,0.11684,0.057975
1,1,-1.869867,12.184925,2020-S1,seche,2020-01-01,2020-06-30,0.057975,-0.081340,1,2020,1,-0.187815,NaN,NaN,0.245790,NaN,0.11684,-0.032191
2,1,-1.924281,12.414088,2020-S1,seche,2020-01-01,2020-06-30,-0.032191,-0.011703,1,2020,1,0.057975,-0.187815,NaN,-0.090166,NaN,0.11684,-0.022204
3,1,-1.925239,12.413330,2020-S1,seche,2020-01-01,2020-06-30,-0.022204,-0.029976,1,2020,1,-0.032191,0.057975,-0.187815,0.009988,0.165612,0.11684,-0.026483
4,1,-1.926437,12.413010,2020-S1,seche,2020-01-01,2020-06-30,-0.026483,-0.033256,1,2020,1,-0.022204,-0.032191,0.057975,-0.004279,-0.084458,0.11684,-0.035727


# 4. Seuils relatifs basés sur la distribution réelle

In [28]:

p33 = df['ndwi_futur'].quantile(0.33)
p66 = df['ndwi_futur'].quantile(0.66)

print(f"\n=== Seuils calibrés ===")
print(f"NDWI min  : {df['ndwi_futur'].min():.4f}")
print(f"Seuil p33 : {p33:.4f} → en dessous = risque élevé")
print(f"Seuil p66 : {p66:.4f} → au dessus  = risque faible")
print(f"NDWI max  : {df['ndwi_futur'].max():.4f}")

def get_risk(ndwi):
    if pd.isna(ndwi):
        return np.nan
    if ndwi >= p66:
        return 0.1
    elif ndwi >= p33:
        return 0.5
    else:
        return 0.9

df['risk_score'] = df['ndwi_futur'].apply(get_risk)

print(f"\n=== Distribution risk_score ===")
print(df['risk_score'].value_counts())


=== Seuils calibrés ===
NDWI min  : -0.6162
Seuil p33 : 0.0410 → en dessous = risque élevé
Seuil p66 : 0.2296 → au dessus  = risque faible
NDWI max  : 0.5891

=== Distribution risk_score ===
risk_score
0.1    945
0.5    917
0.9    917
Name: count, dtype: int64


In [29]:
# 5. Nettoyer
df = df.dropna(subset=['ndwi_t1', 'ndwi_t2', 'ndwi_futur', 'risk_score'])
print(f"\nLignes pour entraînement : {len(df)}")

features = [
    'ndwi_t1',
    'ndwi_t2',
    'ndwi_t3',
    'tendance',
    'tendance_long',
    'ndwi_moy_src',
    'ndvi_moyen',
    'latitude',
    'longitude',
    'saison_encoded',
    'annee',
    'semestre'
]

X = df[features].fillna(0)
y = df['risk_score']




Lignes pour entraînement : 2777


In [30]:
# 6. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [32]:
import os
# 7. Comparer les modèles
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

models = {
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR':               SVR(kernel='rbf', C=1.0),
}

print("\n=== Comparaison des modèles ===")
best_model = None
best_r2    = -999
best_name  = ''

for name, model in models.items():
    if name == 'SVR':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    print(f"{name:25} → MAE: {mae:.4f} | R²: {r2:.4f}")

    if r2 > best_r2:
        best_r2    = r2
        best_model = model
        best_name  = name

print(f"\nMeilleur modèle : {best_name} (R²={best_r2:.4f})")

# 8. Importance features
rf = models['Random Forest']
importances = pd.DataFrame({
    'feature':    features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== Importance des features ===")
print(importances.to_string(index=False))

# 9. Sauvegarder
os.makedirs('app/models', exist_ok=True)
joblib.dump(best_model, 'app/models/watertracker_rf.pkl')
joblib.dump(scaler,     'app/models/scaler.pkl')
joblib.dump(le_saison,  'app/models/encoder_saison.pkl')
joblib.dump({'p33': p33, 'p66': p66}, 'app/models/seuils.pkl')

print(f"\nModèle '{best_name}' sauvegardé ✅")


=== Comparaison des modèles ===
Random Forest             → MAE: 0.1501 | R²: 0.5550
Gradient Boosting         → MAE: 0.1619 | R²: 0.5087
SVR                       → MAE: 0.1729 | R²: 0.4776

Meilleur modèle : Random Forest (R²=0.5550)

=== Importance des features ===
       feature  importance
    ndvi_moyen    0.274130
       ndwi_t1    0.247456
saison_encoded    0.097375
     longitude    0.081471
       ndwi_t3    0.060087
       ndwi_t2    0.059503
      latitude    0.058606
      tendance    0.048923
 tendance_long    0.044321
         annee    0.019671
      semestre    0.008456
  ndwi_moy_src    0.000000

Modèle 'Random Forest' sauvegardé ✅
